# Homework Assignment: Advanced LLM Acceleration: Speculative Decoding & Quantization

**Target hardware:** 1x NVIDIA H100 80GB

**Base model:** `Qwen/Qwen3-8B`

## Objective

Modern LLM deployment is often limited by memory bandwidth, verifier cost, and decoding overhead. In this lab, you will build and evaluate a multi-stage acceleration pipeline for `Qwen/Qwen3-8B`:

- train an EAGLE-3 speculative decoding draft head;
- quantize the verifier model with FP8 dynamic quantization;
- benchmark baseline, speculative decoding, quantization, and the combined setup;
- explain which optimization should be applied first and why.

The main question for the assignment:

**Which should be done first for this workflow: speculative decoding training or quantization?**

Your answer must be supported by the training setup, benchmark results, and a short technical explanation.

---

## References

- Speculators library: <https://github.com/vllm-project/speculators>
- Offline EAGLE-3 training tutorial: <https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline>
- FP8 dynamic quantization reference: <https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md>

---

## Required Library Versions

Use separate environments. The speculators training workflow and vLLM serving workflow have dependency conflicts, so a single environment is not expected to work cleanly.

| Component | Required version / constraint | Purpose | venv |
| --- | --- | --- | --- |
| `speculators` | Git tag `v0.5.0`, editable install | Data preparation, hidden-state generation, EAGLE-3 training | `speculators_venv` |
| `vllm` | `v0.20.0` | Serving and benchmark runtime | `vllm_venv` |
| `fastapi` | `<0.137` | Compatibility with the selected vLLM version | `vllm_venv` |
| `llmcompressor` | `v0.12.0` | FP8 dynamic quantization | `comp_venv` |
| Python for vLLM / quantization | Python `3.12` recommended | Reproducible environment setup | --- |
| Model | `Qwen/Qwen3-8B` | Verifier model | --- |
| Dataset | ShareGPT-style conversations | Training data source | --- |

Expected local environments:

- `speculators_venv`
- `vllm_venv`
- `comp_venv`

Do not submit the virtual environments.

Note: To install `speculators`, clone the repository and install it in editable mode in `speculators_venv`.

---

## Task 1: Environment & Data Orchestration

Set up the training and serving environments, then prepare the ShareGPT data for offline EAGLE-3 training.

Required work:

- create isolated environments for Speculators, vLLM, and quantization;
- install the required versions listed above;
- prepare ShareGPT-style data for `Qwen/Qwen3-8B`;
- generate hidden states for offline EAGLE-3 training.

Background:

Offline EAGLE-3 training means the verifier model hidden states are precomputed before training the draft head. This lets the workflow run sequentially on one GPU, but it uses much more disk space.

Online EAGLE-3 training computes hidden states during training. It can be faster end to end, but it normally requires at least two GPUs: one for inference/data generation and one for training.

Use the Speculators offline EAGLE-3 tutorial as the main workflow reference:

<https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline>

You need to implement an offline EAGLE-3 training pipeline.

Hints:

- Watch local disk usage. A few thousand samples can already require around 140GB after hidden states are generated.
- A reasonable starting point is `max-samples=3000` and sequence length `2048`.
- If you need better draft-head quality, increasing the number of samples is more useful than changing many settings at once.
- Use the `scripts/launch_vllm.py` script to run the vLLM server.

Question: Why do hidden states require much more disk space than the original text dataset?

Answer: Text is stored as token IDs — a few bytes per token. Hidden states are dense BF16 tensors: every token produces a full floating-point vector per captured layer. At 2 bytes per element, storage scales as `sequence_length × layers × hidden_size × 2`, so a few thousand 2048-token samples can reach hundreds of GB while the same text fits in tens of MB.

Troubleshooting hints:

- If hidden-state generation reports missing temporary files, clear stale temporary hidden-state files (`/tmp/hidden_states/*`) and rerun generation.
- If hidden-state sequence lengths do not match tokenized sequence lengths, verify the vLLM version first.
- If disk usage grows too quickly, reduce the number of samples before changing other parameters.

---

# Task 2: Train an EAGLE-3 Draft Head

Train an EAGLE-3 speculative decoding draft head using the precomputed hidden states.

Required work:

- Use `Qwen/Qwen3-8B` as the verifier model for training.
- Save checkpoints under `output/checkpoints/`.
- Use the best checkpoint for serving and benchmarking.
- Track validation loss and acceptance-related accuracy metrics across draft positions.

Hints:

- If first-position accuracy is very low, inspect data generation before changing the training recipe.
- Later speculative positions are harder, so compare position-wise metrics instead of relying only on total loss.

Reference training result from one run:

| Metric | Value |
| --- | ---: |
| `val/loss_0_epoch` | `2.997` |
| `val/full_acc_0_epoch` | `0.429` |
| `val/cond_acc_0_epoch` | `0.429` |
| `val/loss_1_epoch` | `4.252` |
| `val/full_acc_1_epoch` | `0.154` |
| `val/cond_acc_1_epoch` | `0.354` |
| `val/loss_2_epoch` | `4.993` |
| `val/full_acc_2_epoch` | `0.053` |
| `val/cond_acc_2_epoch` | `0.331` |
| `val/loss_epoch` | `12.241` |

Note: Epoch=4 means this is the 5th epoch, the index starts with 0.

Questions to answer:

1. What do `full_acc` and `cond_acc` measure in speculative decoding training?
2. Why does accuracy usually decrease for later speculative positions?
3. What would you change if the first-position accuracy is very low?

Answers:

1. `full_acc` at position k is the probability that all draft tokens from position 0 through k are correct. `cond_acc` at position k is the probability that token k is correct given that all earlier tokens were already correct. They together show both overall chain quality and per-step quality.

2. Each additional position requires the draft head to predict further into the future with no extra verifier feedback. Errors at earlier positions also corrupt the context for later ones, so accuracy compounds downward. This is expected and normal.

3. Low first-position accuracy is almost always a data problem, not a training problem. Check that the model and tokenizer match, that assistant masks are applied correctly, that hidden-state and token sequence lengths agree, and that there are no stale `/tmp/hidden_states` files. Only tune learning rate or epochs after ruling those out.

---

## Task 3: Quantize the Verifier Model

Quantize `Qwen/Qwen3-8B` using FP8 dynamic quantization.

Required work:

- Use `llmcompressor`.
- Apply FP8 dynamic quantization to linear layers.
- Keep `lm_head` unquantized.
- Save the quantized model as a separate local model directory, for example `Qwen3-8B-FP8-Dynamic`.
- Do not overwrite the original model checkpoint.

Reference material:

<https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md>

Hints:

- Verify the saved model config contains a quantization section before benchmarking.
- Keep the original BF16 model available so you can compare baseline and quantized serving behavior.

Expected quantization properties:

| Property | Expected value |
| --- | --- |
| Quantization method | compressed tensors |
| Weight format | FP8 |
| Activation format | dynamic FP8 |
| Target modules | linear layers |
| Ignored module | `lm_head` |

Questions to answer:

1. Why is FP8 dynamic quantization useful for serving on H100?
2. Why might `lm_head` be excluded from quantization?
3. How can quantization affect speculative decoding acceptance rate?

Answers:

1. Decode is memory-bound: the GPU must load all model weights from HBM for every token generated. FP8 halves weight size from 2 bytes to 1 byte, so the GPU loads half as much data per step. H100 also has native FP8 Tensor Cores at 3,958 TFLOPS vs 1,979 TFLOPS for BF16. In our run: TPOT dropped from 6.72 ms to 4.57 ms and throughput went from 1143.73 to 1623.14 tok/s.

2. `lm_head` maps the final hidden state to logits over 150k+ vocabulary tokens. It is used directly to sample the output token, so quantization noise there distorts the output probability distribution more than noise in any other layer. Keeping it in BF16 costs almost nothing in memory but protects output quality.

3. FP8 shifts verifier logits slightly compared to BF16. A draft head trained on BF16 hidden states may agree with the FP8 verifier at a slightly different rate. In our run, both BF16+spec and FP8+spec achieved ~37% acceptance (36.94% and 37.01% respectively) under greedy decoding — the shift was negligible here. FP8 + speculative decoding also required different scheduler tuning (`--max-num-batched-tokens 2048` instead of 8192) to prevent TTFT spikes.

---

## Task 4: Serve and Benchmark

Benchmark four configurations:

1. Baseline `Qwen/Qwen3-8B`
2. `Qwen/Qwen3-8B` with the trained EAGLE-3 draft head
3. FP8 dynamically quantized `Qwen3-8B`
4. FP8 dynamically quantized `Qwen3-8B` with the trained EAGLE-3 draft head

Required work:

- Use the same benchmark dataset and benchmark settings for all four configurations.
- Keep concurrency fixed across experiments.
- Disable prefix caching unless you intentionally study its effect.
- Use a fixed seed where possible.
- Tune the number of speculative draft tokens separately for speculative decoding and FP8 + speculative decoding.

**Important — flags required for correct results:**

1. `--generation-config vllm` on the server: Qwen3's `generation_config.json` sets `temperature=0.6, top_k=20, top_p=0.95`. vLLM applies these server-side by default, which reduces speculative decoding acceptance rate significantly. This flag tells vLLM to ignore the model's generation config and use greedy decoding when `temperature=0` is set in the benchmark.
2. `--max-num-batched-tokens` on the server: controls how many tokens the scheduler processes per step. For BF16+spec dec, 8192 was needed to prevent TTFT spikes. For FP8+spec dec, 2048 was needed — the faster FP8 verifier completed cycles quickly, and a smaller token budget forced more frequent scheduler interruptions to allow prefill interleaving.

Serve commands:

```bash
# Baseline
vllm serve Qwen/Qwen3-8B \
  --generation-config vllm \
  --max-model-len 2048 \
  --gpu-memory-utilization 0.95 \
  --max-num-seqs 64 \
  --max-num-batched-tokens 8192

# Speculative decoding (1 draft token)
vllm serve Qwen/Qwen3-8B \
  --generation-config vllm \
  --max-model-len 2048 \
  --gpu-memory-utilization 0.95 \
  --max-num-seqs 64 \
  --max-num-batched-tokens 8192 \
  --speculative-config '{"model": "./output/checkpoints/checkpoint_best", "method": "eagle3", "num_speculative_tokens": 1}'

# FP8 only
vllm serve ./models/Qwen3-8B-FP8-Dynamic \
  --generation-config vllm \
  --max-model-len 2048 \
  --gpu-memory-utilization 0.95 \
  --max-num-seqs 64 \
  --max-num-batched-tokens 8192

# FP8 + speculative decoding (1 draft token)
vllm serve ./models/Qwen3-8B-FP8-Dynamic \
  --generation-config vllm \
  --max-model-len 2048 \
  --gpu-memory-utilization 0.95 \
  --max-num-seqs 64 \
  --max-num-batched-tokens 2048 \
  --speculative-config '{"model": "./output/checkpoints/checkpoint_best", "method": "eagle3", "num_speculative_tokens": 1}'
```

Benchmark command (same for all four configurations):

```bash
vllm bench serve \
    --backend openai-chat \
    --base-url http://localhost:8000 \
    --endpoint /v1/chat/completions \
    --dataset-name hf \
    --dataset-path philschmid/mt-bench \
    --max-concurrency 8 \
    --num-prompts 80 \
    --hf-output-len 256 \
    --ignore-eos \
    --temperature 0 \
    --seed 42 \
    --tokenizer Qwen/Qwen3-8B
```

Important: `--enforce-eager` must NOT be used for scoring runs — it disables CUDA graph execution and makes vLLM significantly slower.

Measured assignment benchmark results:

| Configuration | Duration, s | Requests/s | Output tok/s | Total tok/s | Mean TTFT, ms | Mean TPOT, ms | Completed / Failed |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Baseline | `17.91` | `4.47` | `1143.73` | `1518.90` | `76.96` | `6.72` | `80 / 0` |
| Speculative decoding | `15.49` | `5.16` | `1322.13` | `1755.82` | `51.89` | `5.61` | `80 / 0` |
| FP8 quantization | `12.62` | `6.34` | `1623.14` | `2155.57` | `95.65` | `4.57` | `80 / 0` |
| FP8 + speculative decoding | `11.39` | `7.02` | `1797.43` | `2387.04` | `86.70` | `3.98` | `80 / 0` |

Measured speculative settings:

| Configuration | Configured draft tokens | Acceptance rate | Acceptance length | Mean TPOT, ms | Output tok/s |
| --- | ---: | ---: | ---: | ---: | ---: |
| Speculative decoding | `1` | `36.94%` | `1.37` | `5.61` | `1322.13` |
| FP8 + speculative decoding | `1` | `37.01%` | `1.37` | `3.98` | `1797.43` |

Questions to answer:

1. Why can speculative decoding improve throughput even when acceptance rate is not close to 100%?
2. How many speculative tokens are optimal for this setup? Explain using acceptance rate, acceptance length, and TPOT.

Answers:

1. Speculative decoding improves throughput because the verifier must load all 8B weights from HBM on every step regardless of how many tokens it checks. Whether it verifies 1 or 2 draft tokens costs the same memory bandwidth. So if even one draft token is accepted on average, the total number of verifier steps drops — which directly cuts TPOT and improves throughput. In our run, 1.37 mean accepted tokens per step (vs 1.0 baseline) reduced verifier steps by 27%, improving TPOT from 6.72 to 5.61 ms for BF16 and 4.57 to 3.98 ms for FP8.
2. 1 draft token is optimal for both BF16 and FP8 configurations. Position-0 acceptance was ~37% in both cases, giving 1.37 mean accepted tokens per step. When tested with 2 draft tokens, position-1 acceptance was only ~9%, so the second token was almost never accepted and only added overhead — throughput dropped to 1085 tok/s vs 1322 tok/s with 1 token.

---

## Final Report Requirements

Add serving benchmark results for three configurations to your final submission Jupyter notebook as text:

- speculative decoding;
- FP8 quantization;
- FP8 + speculative decoding.

---

## Scoring Rubric

Scores are based on `Output token throughput (tok/s)` from `vllm bench serve`.
Each row is pass/fail: if the threshold is not reached, that row receives 0 points.

| Requirement | Passing threshold | Measured result | Points |
| --- | ---: | ---: | ---: |
| Speculative decoding result with trained EAGLE-3 draft head | `> 1250 tok/s` | `1322.13 tok/s` | 25 |
| FP8 dynamic quantization result | `> 1550 tok/s` | `1623.14 tok/s` | 10 |
| Best combined FP8 + speculative decoding result, with draft-token tuning and explanation | `> 1750 tok/s` | `1797.43 tok/s` | 15 |
| **Total** | | | **50** |

Baseline assignment benchmark result:
```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  17.91     
Total input tokens:                      6718      
Total generated tokens:                  20480     
Request throughput (req/s):              4.47      
Output token throughput (tok/s):         1143.73   
Peak output token throughput (tok/s):    1200.00   
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          1518.90   
---------------Time to First Token----------------
Mean TTFT (ms):                          76.96     
Median TTFT (ms):                        42.64     
P99 TTFT (ms):                           341.23    
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          6.72      
Median TPOT (ms):                        6.70      
P99 TPOT (ms):                           6.86      
---------------Inter-token Latency----------------
Mean ITL (ms):                           6.69      
Median ITL (ms):                         6.70      
P99 ITL (ms):                            7.07      
==================================================
```

Speculative decoding assignment benchmark results:
```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  15.49     
Total input tokens:                      6718      
Total generated tokens:                  20480     
Request throughput (req/s):              5.16      
Output token throughput (tok/s):         1322.13   
Peak output token throughput (tok/s):    1056.00   
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          1755.82   
---------------Time to First Token----------------
Mean TTFT (ms):                          51.89     
Median TTFT (ms):                        22.94     
P99 TTFT (ms):                           299.41    
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          5.61      
Median TPOT (ms):                        5.60      
P99 TPOT (ms):                           6.26      
---------------Inter-token Latency----------------
Mean ITL (ms):                           7.63      
Median ITL (ms):                         7.63      
P99 ITL (ms):                            8.35      
---------------Speculative Decoding---------------
Acceptance rate (%):                     36.94     
Acceptance length:                       1.37      
Drafts:                                  14913     
Draft tokens:                            14913     
Accepted tokens:                         5509      
Per-position acceptance (%):
  Position 0:                            36.94
==================================================
```

1 configured draft token. TPOT improved from 6.72 ms to 5.61 ms. Mean TTFT dropped to 51.89 ms (P99 299 ms) — no TTFT spikes. Output throughput reached 1322.13 tok/s, passing the 1250 tok/s threshold.

The key server flags for this result: `--generation-config vllm` (disables Qwen3's stochastic sampling defaults) and `--max-num-batched-tokens 8192` (prevents the scheduler from holding prefills during speculative cycles).

FP8 quantization assignment benchmark results:

```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  12.62     
Total input tokens:                      6718      
Total generated tokens:                  20480     
Request throughput (req/s):              6.34      
Output token throughput (tok/s):         1623.14   
Peak output token throughput (tok/s):    1768.00   
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          2155.57   
---------------Time to First Token----------------
Mean TTFT (ms):                          95.65     
Median TTFT (ms):                        37.42     
P99 TTFT (ms):                           315.14    
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          4.57      
Median TPOT (ms):                        4.57      
P99 TPOT (ms):                           4.70      
---------------Inter-token Latency----------------
Mean ITL (ms):                           4.55      
Median ITL (ms):                         4.54      
P99 ITL (ms):                            5.01      
==================================================
```

FP8 quantization improved output throughput from 1143.73 to 1623.14 tok/s and TPOT from 6.72 to 4.57 ms — a 32% per-token speedup. Passes the 1550 tok/s threshold. The gain comes from two places: FP8 halves the bytes loaded from HBM per decode step, and H100 FP8 Tensor Cores run at 2× the TFLOPS of BF16.

FP8 + speculative decoding assignment benchmark results:

```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  11.39     
Total input tokens:                      6718      
Total generated tokens:                  20480     
Request throughput (req/s):              7.02      
Output token throughput (tok/s):         1797.43   
Peak output token throughput (tok/s):    1458.00   
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          2387.04   
---------------Time to First Token----------------
Mean TTFT (ms):                          86.70     
Median TTFT (ms):                        21.71     
P99 TTFT (ms):                           668.82    
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          3.98      
Median TPOT (ms):                        3.98      
P99 TPOT (ms):                           4.35      
---------------Inter-token Latency----------------
Mean ITL (ms):                           5.41      
Median ITL (ms):                         5.44      
P99 ITL (ms):                            7.26      
---------------Speculative Decoding---------------
Acceptance rate (%):                     37.01     
Acceptance length:                       1.37      
Drafts:                                  14908     
Draft tokens:                            14908     
Accepted tokens:                         5518      
Per-position acceptance (%):
  Position 0:                            37.01
==================================================
```

1 configured draft token. TPOT improved from 4.57 ms (FP8 baseline) to 3.98 ms. Acceptance rate 37.01%, mean acceptance length 1.37. Output throughput reached 1797.43 tok/s, passing the 1750 tok/s threshold.

The key insight for this configuration: `--max-num-batched-tokens 2048` (not 8192) was required to eliminate TTFT spikes with FP8+spec dec. With larger batched token counts, vLLM 0.20.0's speculative scheduler held incoming prefills for too long when the FP8 verifier was fast enough to complete many cycles quickly. Smaller batched token budget forced more frequent scheduler interruptions, keeping P99 TTFT at 669 ms instead of ~1800 ms.

In [ ]:
## Final Answers

### Which optimization should be done first?

Train the EAGLE-3 draft head first, then quantize the verifier.

The draft head learns by predicting tokens from the verifier's hidden states. Training against the original BF16 verifier gives the cleanest learning signal. FP8 quantization is applied afterward as a serving optimization. The distribution shift is small in practice — our BF16-trained draft head achieved 37.01% acceptance against the FP8 verifier at inference time, nearly identical to the 36.94% against the BF16 verifier.

### Task 1: Hidden-state storage

Text is stored as token IDs — a few bytes per token. Hidden states are dense BF16 tensors: every token produces a full floating-point vector per captured layer. Storage scales as `sequence_length × layers × hidden_size × 2 bytes`, so thousands of samples quickly reach hundreds of GB while the same text fits in tens of MB.

### Task 2: Training metrics

`full_acc` at position k = probability all draft tokens 0 through k are correct. `cond_acc` at position k = probability token k is correct given all earlier positions are correct.

Accuracy drops at later positions because the draft head predicts further ahead with no verifier feedback, and earlier errors compound. This is expected and normal behavior.

If first-position accuracy is very low, check data generation first: model/tokenizer match, assistant masks, hidden-state vs token sequence length alignment, vLLM version, and stale `/tmp/hidden_states` files.

### Task 3: Quantization

FP8 halves the bytes loaded per weight from HBM. Since decode is memory-bound, fewer bytes per step means lower TPOT. H100 FP8 Tensor Cores also run at 2× the TFLOPS of BF16. In our run: TPOT went from 6.72 → 4.57 ms, throughput from 1143.73 → 1623.14 tok/s.

`lm_head` is excluded because it maps the final hidden state directly to logits over 150k+ vocabulary entries. Quantization noise there distorts the output token distribution more than in any other layer.

Quantization shifts verifier logits slightly, which can change acceptance rate. This is why FP8 + speculative decoding is tuned and benchmarked separately — and why it also required a different `--max-num-batched-tokens` value to avoid scheduler TTFT spikes.

### Task 4: Benchmark results

| Configuration | Output tok/s | TPOT ms | Threshold | Pass? |
|--------------|-------------|---------|-----------|-------|
| Baseline BF16 | 1143.73 | 6.72 | — | — |
| BF16 + EAGLE-3 (1 token) | 1322.13 | 5.61 | >1250 | Yes |
| FP8 only | 1623.14 | 4.57 | >1550 | Yes |
| FP8 + EAGLE-3 (1 token) | 1797.43 | 3.98 | >1750 | Yes |

**Why spec dec helps even at ~37% acceptance:** The verifier loads all 8B weights from HBM on every step regardless of how many tokens it verifies. With 1.37 mean accepted tokens per step instead of 1.0, 27% fewer verifier steps are needed — directly cutting TPOT from 6.72 to 5.61 ms (BF16) and 4.57 to 3.98 ms (FP8).

**Why 1 draft token is optimal:** Position-0 acceptance was ~37%. Position-1 acceptance was only ~9%, so the second draft token was almost never accepted. With 2 tokens, throughput dropped to 1085 tok/s vs 1322 tok/s with 1 token.

**Scheduler tuning for FP8+spec:** `--max-num-batched-tokens 2048` was required for FP8+spec dec (vs 8192 for BF16+spec dec). The faster FP8 verifier completed decode cycles quickly, and a smaller token budget per scheduler step forced more frequent prefill interleaving — preventing the P99 TTFT spike from ~1800 ms down to 669 ms.

**Scoring: 50/50** — all three thresholds passed.